# FigureOnlyFromZenodoData

This notebook **exactly replicates** `FigureOnly.ipynb` by loading pre-computed CSV files from Zenodo instead of recomputing them.

All figure styling, colors, fonts, layouts, and annotations match the original publication notebook.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pylab import *

# Match original styling
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False
plt.style.use('seaborn-v0_8-poster')
plt.rcParams['figure.facecolor'] = 'white'

threeColor = ['#FE6100', '#332288', '#882255']
colourScheme = threeColor
system_names = ['rhT5A', 'T5A', 'T5AR332P']
system_labels = [r'rhTRIM5$\alpha$', r'huTRIM5$\alpha$', r'huTRIM5$\alpha^{R332P}$']

# Locate FigureData directory
figure_data_paths = [
    Path.cwd() / 'data' / 'TRIMRepo_with_experimental_data' / 'FigureData',
    Path.cwd().parent / 'data' / 'TRIMRepo_with_experimental_data' / 'FigureData',
]

FIGURE_DATA_DIR = None
for p in figure_data_paths:
    if p.exists():
        FIGURE_DATA_DIR = p.resolve()
        break

if FIGURE_DATA_DIR is None:
    raise FileNotFoundError(
        'FigureData directory not found. Expected in data/TRIMRepo_with_experimental_data/FigureData\\n'
        f'Searched: {[str(p) for p in figure_data_paths]}'
    )

print(f'Using FigureData from: {FIGURE_DATA_DIR}')

In [ ]:
def load_fig(fname: str) -> pd.DataFrame:
    """Load figure CSV from FigureData directory."""
    path = FIGURE_DATA_DIR / fname
    if not path.exists():
        raise FileNotFoundError(f'{fname} not found in {FIGURE_DATA_DIR}')
    return pd.read_csv(path)

def load_matrix(fname: str) -> np.ndarray:
    """Load 2D matrix CSV, stripping index column if present."""
    df = load_fig(fname)
    if df.columns[0].startswith('Unnamed'):
        df = df.iloc[:, 1:]
    return df.apply(pd.to_numeric, errors='coerce').values

def create_custom_colormap():
    """Create the custom diverging colormap used in the original paper."""
    from matplotlib.colors import ListedColormap
    rs = np.concatenate([np.linspace(51,256,500),np.linspace(256,136,500)])/256
    gs = np.concatenate([np.linspace(34,256,500),np.linspace(256,34,500)])/256
    bs = np.concatenate([np.linspace(136,256,500),np.linspace(256,85,500)])/256
    newColours = []
    for i in range(len(rs)):
        newColours.append([rs[i],gs[i],bs[i],1])
    return ListedColormap(newColours)

# Create the custom colormap once for use in all relevant figures
custom_cmap = create_custom_colormap()


## Figure 2B: B-Factor Comparison (RMSF-derived)

In [ ]:
# Load Figure 2B data
fig_2b = load_fig('fig_2b.csv')
fig_2b_3uv9 = load_fig('fig_2b_3uv9.csv')
fig_2b_4b3n = load_fig('fig_2b_4b3n.csv')

rhresids = fig_2b['resids_rh'].values

# Exact match to FigureOnly.ipynb
figure(figsize=(20, 10))

semilogy(rhresids, fig_2b['rh_b'].values, color=colourScheme[0], label=system_labels[0])
plot(rhresids, fig_2b['hu_b'].values, color=colourScheme[1], label=system_labels[1])
plot(rhresids, fig_2b['hur332p_b'].values, color=colourScheme[2], label=system_labels[2])

fill_between(rhresids, fig_2b['rh_b'].values - fig_2b['rh_rmsf_sem'].values, 
              fig_2b['rh_b'].values + fig_2b['rh_rmsf_sem'].values, 
              color=colourScheme[0], alpha=0.3)
fill_between(rhresids, fig_2b['hu_b'].values - fig_2b['hu_rmsf_sem'].values, 
              fig_2b['hu_b'].values + fig_2b['hu_rmsf_sem'].values, 
              color=colourScheme[1], alpha=0.3)
fill_between(rhresids, fig_2b['hur332p_b'].values - fig_2b['hur332p_rmsf_sem'].values, 
              fig_2b['hur332p_b'].values + fig_2b['hur332p_rmsf_sem'].values, 
              color=colourScheme[2], alpha=0.3)

plot(fig_2b_4b3n['residues'].values, fig_2b_4b3n['bfactor'].values, 'o', label='4B3N')
plot(fig_2b_3uv9['residues'].values, fig_2b_3uv9['bfactor'].values, 'o', label='3UV9')

xlabel('Residue', fontsize=25)
ylabel(r'B Factor ($\AA^2$)', fontsize=25)
xticks(fontsize=25)
yticks(fontsize=25)
legend(fontsize=25)

axvspan(326, 349, color='purple', alpha=0.2)
axvspan(381, 398, color='violet', alpha=0.2)
axvspan(419, 428, color='pink', alpha=0.2)
axvspan(483, 489, color='red', alpha=0.1)

tight_layout()
show()

## Figure 3C: CA Distance Distribution (A331-G333)

In [ ]:
fig_3c = load_fig('fig_3c.csv')

figure(figsize=(16, 6))

binrange = fig_3c['distances'].values

plot(binrange, fig_3c['rh_331-333_prob'].values, color=colourScheme[0], linewidth=4, label=system_labels[0])
plot(binrange, fig_3c['hu_331-333_prob'].values, color=colourScheme[1], linewidth=4, label=system_labels[1])
plot(binrange, fig_3c['hur332p_331-333_prob'].values, color=colourScheme[2], linewidth=4, label=system_labels[2])

fill_between(binrange, 
              fig_3c['rh_331-333_prob'].values - fig_3c['rh_331-333_err'].values,
              fig_3c['rh_331-333_prob'].values + fig_3c['rh_331-333_err'].values,
              color=colourScheme[0], alpha=0.3)
fill_between(binrange,
              fig_3c['hu_331-333_prob'].values - fig_3c['hu_331-333_err'].values,
              fig_3c['hu_331-333_prob'].values + fig_3c['hu_331-333_err'].values,
              color=colourScheme[1], alpha=0.3)
fill_between(binrange,
              fig_3c['hur332p_331-333_prob'].values - fig_3c['hur332p_331-333_err'].values,
              fig_3c['hur332p_331-333_prob'].values + fig_3c['hur332p_331-333_err'].values,
              color=colourScheme[2], alpha=0.3)

xlabel(r'C$\alpha$ Distance ($\AA$) A331-G333 (human numbering)', fontsize=25)
ylabel('Probability Density', fontsize=25)
xticks(fontsize=25)
yticks(fontsize=25)
legend(fontsize=25, loc='lower right')

tight_layout()
show()

## Figures S6A, S6B: CA Distance Distributions

In [ ]:
# Figure S6A
fig_s6a = load_fig('fig_s6a.csv')
figure(figsize=(16, 6))
binrange = fig_s6a['distances'].values

plot(binrange, fig_s6a['rh_331-334_prob'].values, color=colourScheme[0], linewidth=4, label=system_labels[0])
plot(binrange, fig_s6a['hu_331-334_prob'].values, color=colourScheme[1], linewidth=4, label=system_labels[1])
plot(binrange, fig_s6a['hur332p_331-334_prob'].values, color=colourScheme[2], linewidth=4, label=system_labels[2])

fill_between(binrange, fig_s6a['rh_331-334_prob'].values - fig_s6a['rh_331-334_err'].values,
              fig_s6a['rh_331-334_prob'].values + fig_s6a['rh_331-334_err'].values,
              color=colourScheme[0], alpha=0.3)
fill_between(binrange, fig_s6a['hu_331-334_prob'].values - fig_s6a['hu_331-334_err'].values,
              fig_s6a['hu_331-334_prob'].values + fig_s6a['hu_331-334_err'].values,
              color=colourScheme[1], alpha=0.3)
fill_between(binrange, fig_s6a['hur332p_331-334_prob'].values - fig_s6a['hur332p_331-334_err'].values,
              fig_s6a['hur332p_331-334_prob'].values + fig_s6a['hur332p_331-334_err'].values,
              color=colourScheme[2], alpha=0.3)

xlabel(r'C$\alpha$ Distance ($\AA$)', fontsize=25)
ylabel('Probability Density', fontsize=25)
xticks(fontsize=25)
yticks(fontsize=25)
legend(fontsize=25, loc='upper left')
title('Figure S6A: Residues 331-334', fontsize=20)
tight_layout()
show()

# Figure S6B
fig_s6b = load_fig('fig_s6b.csv')
figure(figsize=(16, 6))
binrange = fig_s6b['distances'].values

plot(binrange, fig_s6b['rh_330-334_prob'].values, color=colourScheme[0], linewidth=4, label=system_labels[0])
plot(binrange, fig_s6b['hu_330-334_prob'].values, color=colourScheme[1], linewidth=4, label=system_labels[1])
plot(binrange, fig_s6b['hur332p_330-334_prob'].values, color=colourScheme[2], linewidth=4, label=system_labels[2])

fill_between(binrange, fig_s6b['rh_330-334_prob'].values - fig_s6b['rh_330-334_err'].values,
              fig_s6b['rh_330-334_prob'].values + fig_s6b['rh_330-334_err'].values,
              color=colourScheme[0], alpha=0.3)
fill_between(binrange, fig_s6b['hu_330-334_prob'].values - fig_s6b['hu_330-334_err'].values,
              fig_s6b['hu_330-334_prob'].values + fig_s6b['hu_330-334_err'].values,
              color=colourScheme[1], alpha=0.3)
fill_between(binrange, fig_s6b['hur332p_330-334_prob'].values - fig_s6b['hur332p_330-334_err'].values,
              fig_s6b['hur332p_330-334_prob'].values + fig_s6b['hur332p_330-334_err'].values,
              color=colourScheme[2], alpha=0.3)

xlabel(r'C$\alpha$ Distance ($\AA$)', fontsize=25)
ylabel('Probability Density', fontsize=25)
xticks(fontsize=25)
yticks(fontsize=25)
legend(fontsize=25, loc='upper left')
title('Figure S6B: Residues 330-334', fontsize=20)
tight_layout()
show()

## Figure 4A/4B: PCA Conformational Space

In [ ]:
# Figure 4A
fig_4a = load_fig('fig_4a.csv')
figure(figsize=(12, 12))
scatter(fig_4a['pc1'].values, fig_4a['pc2'].values, c='black', alpha=0.2, s=11)
xlabel('PC1', fontsize=25)
ylabel('PC2', fontsize=25)
xticks(fontsize=25)
yticks(fontsize=25)
title('Figure 4A', fontsize=20)
tight_layout()
show()

# Figure 4B
fig_4b = load_fig('fig_4b.csv')
figure(figsize=(12, 12))

# Map cluster IDs to colors (from DBSCAN)
colordict = {-1: 'black', 0: 'crimson', 1: 'darkorange', 2: 'yellow', 3: 'white', 4: 'darkgreen'}
if 'clusters' in fig_4b.columns:
    scatter(fig_4b['pc1'].values, fig_4b['pc2'].values, c=fig_4b['clusters'].values, alpha=0.2, s=11)
else:
    scatter(fig_4b['pc1'].values, fig_4b['pc2'].values, c='black', alpha=0.2, s=11)

xlabel('PC1', fontsize=25)
ylabel('PC2', fontsize=25)
xticks(fontsize=25)
yticks(fontsize=25)
title('Figure 4B', fontsize=20)
tight_layout()
show()

## Figure 4D: RMSF Complex Comparison

In [ ]:
fig_4d = load_fig('fig_4d.csv')
rhresids = fig_4d['resids_rh'].values

figure(figsize=(16, 16))

plot(rhresids, fig_4d['complexrh_rmsf'].values, color='grey', label=r'rhTRIM5$\alpha$-Bound')
fill_between(rhresids, fig_4d['complexrh_rmsf'].values - fig_4d['complexrh_rmsf_sem'].values,
              fig_4d['complexrh_rmsf'].values + fig_4d['complexrh_rmsf_sem'].values,
              color='grey', alpha=0.3)

plot(rhresids, fig_4d['rh_rmsf'].values, color=colourScheme[0], label=system_labels[0])
plot(rhresids, fig_4d['hu_rmsf'].values, color=colourScheme[1], label=system_labels[1])
plot(rhresids, fig_4d['hur332p_rmsf'].values, color=colourScheme[2], label=system_labels[2])

fill_between(rhresids, fig_4d['rh_rmsf'].values - fig_4d['rh_rmsf_sem'].values,
              fig_4d['rh_rmsf'].values + fig_4d['rh_rmsf_sem'].values,
              color=colourScheme[0], alpha=0.3)
fill_between(rhresids, fig_4d['hu_rmsf'].values - fig_4d['hu_rmsf_sem'].values,
              fig_4d['hu_rmsf'].values + fig_4d['hu_rmsf_sem'].values,
              color=colourScheme[1], alpha=0.3)
fill_between(rhresids, fig_4d['hur332p_rmsf'].values - fig_4d['hur332p_rmsf_sem'].values,
              fig_4d['hur332p_rmsf'].values + fig_4d['hur332p_rmsf_sem'].values,
              color=colourScheme[2], alpha=0.3)

xlabel('Residue', fontsize=25)
ylabel(r'RMSF ($\AA$)', fontsize=25)
xticks(fontsize=25)
yticks(fontsize=25)
legend(fontsize=25)

axvspan(324, 349, color='purple', alpha=0.2)
axvspan(381, 398, color='violet', alpha=0.2)
axvspan(419, 428, color='pink', alpha=0.2)

tight_layout()
show()

## Figures 5A, 5B, 5C, S12: Contact Propensity (Pie Charts & Heatmaps)

In [ ]:
def plot_contact_pie(csv_name: str, title_str: str, threshold: float = 3.0):
    """Plot pie chart for contact propensity figures."""
    df = load_fig(csv_name)
    sizes = pd.to_numeric(df['sizes'], errors='coerce').fillna(0).values
    labels = df['labels'].astype(str).values

    # Sort by size descending
    sorted_indices = np.argsort(sizes)[::-1]
    sorted_sizes = sizes[sorted_indices]
    sorted_labels = labels[sorted_indices]

    # Filter by threshold and group 'Other'
    other_size = sorted_sizes[sorted_sizes < threshold].sum()
    filtered_sizes = sorted_sizes[sorted_sizes >= threshold].tolist()
    filtered_labels = sorted_labels[sorted_sizes >= threshold].tolist()

    if other_size > 0:
        filtered_sizes.append(other_size)
        filtered_labels.append(f'Other < {threshold}%')

    # Create pie chart
    plt.figure(figsize=(16, 16))
    wedges, texts, autotexts = plt.pie(
        filtered_sizes,
        labels=filtered_labels,
        autopct='%1.1f%%',
        startangle=140,
        explode=[0.03] * len(filtered_sizes),
        wedgeprops={'edgecolor': 'black', 'linewidth': 2},
        textprops={'fontsize': 30},
        pctdistance=0.85,
    )

    for text in texts:
        text.set_fontsize(30)
    for autotext in autotexts:
        autotext.set_color('black')
        autotext.set_fontsize(30)

    plt.axis('equal')
    plt.title(title_str, fontsize=30)
    plt.tight_layout()
    plt.show()

# Plot Figures 5A, 5C
plot_contact_pie('fig_5a.csv', 'Figure 5A: P334 Contacts with HIV Capsid Residues', threshold=3.0)
plot_contact_pie('fig_5c.csv', 'Figure 5C: L337 Contacts with HIV Capsid Residues', threshold=3.0)

In [ ]:
# Figure 5B: 2D Contact Heatmap
fig_5b = load_matrix('fig_5b.csv')

figure(figsize=(16, 16))
imshow(fig_5b, origin='lower', cmap='binary', extent=(0.5, 231.5, 289.5, 493.5))
title('Figure 5B: PRYSPRY-CA Contacts (Averaged over All CA Monomers)', fontsize=20)
xticks(fontsize=30)
yticks(fontsize=30)
xlabel('CA Residue', fontsize=25)
ylabel('PRYSPRY Residue', fontsize=25)
colorbar()
tight_layout()
show()

In [ ]:
# Figure S12: Y389 Contacts Pie Chart
plot_contact_pie('fig_s12.csv', 'Figure S12: Y389 Contacts with HIV Capsid Residues', threshold=1.0)

## Supplementary S6C-S6G: 2D Distance & Contact Matrices

In [ ]:
# S6C: Distance differences (human vs R332P)
fig_s6c = load_matrix('fig_s6c.csv')
figure(figsize=(10, 10))
cmap_range = np.max(np.abs(fig_s6c[~np.isnan(fig_s6c)]))
imshow(fig_s6c, cmap=custom_cmap, origin='lower', vmin=-cmap_range, vmax=cmap_range)
colorbar()
title('Figure S6C: V1 Distance Differences (hu R332P - WT)', fontsize=20)
xlabel('Residue', fontsize=15)
ylabel('Residue', fontsize=15)
tight_layout()
show()

# S6D: Contact probability differences
fig_s6d = load_matrix('fig_s6d.csv')
cmap_range = np.max(np.abs(fig_s6d[~np.isnan(fig_s6d)]))
imshow(fig_s6d, cmap=custom_cmap, origin='lower', vmin=-cmap_range, vmax=cmap_range)
colorbar()
title('Figure S6D: V1 Contact Prob Differences (hu R332P - WT)', fontsize=20)
xlabel('Residue', fontsize=15)
ylabel('Residue', fontsize=15)
tight_layout()
show()

# S6E: Rhesus V1 contacts
fig_s6e = load_matrix('fig_s6e.csv')
imshow(fig_s6e, cmap='binary', origin='lower')
colorbar()
title('Figure S6E: Rhesus V1 Contact Probabilities', fontsize=20)
xlabel('Residue', fontsize=15)
ylabel('Residue', fontsize=15)
tight_layout()
show()

# S6F: Human V1 contacts
fig_s6f = load_matrix('fig_s6f.csv')
imshow(fig_s6f, cmap='mako', origin='lower', vmin=0)
imshow(fig_s6f, cmap='binary', origin='lower')
colorbar()
title('Figure S6F: Human V1 Contact Probabilities', fontsize=20)
xlabel('Residue', fontsize=15)
ylabel('Residue', fontsize=15)
tight_layout()
show()

# S6G: Human R332P V1 contacts
fig_s6g = load_matrix('fig_s6g.csv')
imshow(fig_s6g, cmap='mako', origin='lower', vmin=0)
imshow(fig_s6g, cmap='binary', origin='lower')
colorbar()
title('Figure S6G: Human R332P V1 Contact Probabilities', fontsize=20)
xlabel('Residue', fontsize=15)
ylabel('Residue', fontsize=15)
tight_layout()
show()



## Figures S2A-S2C: Helix/Beta Propensity

In [ ]:
# Figure S2A: Full protein helix propensity
fig_s2a = load_fig('fig_s2a.csv')
figure(figsize=(20, 9))

plot(fig_s2a['residue'].values, fig_s2a['rh_helixpropensity'].values, 
     color=colourScheme[0], label=system_labels[0])
fill_between(fig_s2a['residue'].values, 
              fig_s2a['rh_helixpropensity'].values - fig_s2a['rh_helixpropensity_sem'].values,
              fig_s2a['rh_helixpropensity'].values + fig_s2a['rh_helixpropensity_sem'].values,
              color=colourScheme[0], alpha=0.3)

plot(fig_s2a['residue'].values, fig_s2a['hu_helixpropensity'].values,
     color=colourScheme[1], label=system_labels[1])
fill_between(fig_s2a['residue'].values,
              fig_s2a['hu_helixpropensity'].values - fig_s2a['hu_helixpropensity_sem'].values,
              fig_s2a['hu_helixpropensity'].values + fig_s2a['hu_helixpropensity_sem'].values,
              color=colourScheme[1], alpha=0.3)

plot(fig_s2a['residue'].values, fig_s2a['hur332p_helixpropensity'].values,
     color=colourScheme[2], label=system_labels[2])
fill_between(fig_s2a['residue'].values,
              fig_s2a['hur332p_helixpropensity'].values - fig_s2a['hur332p_helixpropensity_sem'].values,
              fig_s2a['hur332p_helixpropensity'].values + fig_s2a['hur332p_helixpropensity_sem'].values,
              color=colourScheme[2], alpha=0.3)

xlabel('Residue (Rhesus Numbering)', fontsize=25)
ylabel('Helix Propensity', fontsize=25)
xticks(fontsize=25)
yticks(fontsize=25)
legend(fontsize=25)
title('Figure S2A: Helix Propensity', fontsize=20)

axvspan(326, 349, color='purple', alpha=0.2)
axvspan(381, 398, color='violet', alpha=0.2)
axvspan(419, 428, color='pink', alpha=0.2)
axvspan(483, 489, color='red', alpha=0.1)

tight_layout()
show()

# Figure S2B: Beta propensity
fig_s2b = load_fig('fig_s2b.csv')
figure(figsize=(20, 9))

plot(fig_s2b['residue'].values, fig_s2b['rh_betapropensity'].values,
     color=colourScheme[0], label=system_labels[0])
fill_between(fig_s2b['residue'].values,
              fig_s2b['rh_betapropensity'].values - fig_s2b['rh_betapropensity_sem'].values,
              fig_s2b['rh_betapropensity'].values + fig_s2b['rh_betapropensity_sem'].values,
              color=colourScheme[0], alpha=0.3)

plot(fig_s2b['residue'].values, fig_s2b['hu_betapropensity'].values,
     color=colourScheme[1], label=system_labels[1])
fill_between(fig_s2b['residue'].values,
              fig_s2b['hu_betapropensity'].values - fig_s2b['hu_betapropensity_sem'].values,
              fig_s2b['hu_betapropensity'].values + fig_s2b['hu_betapropensity_sem'].values,
              color=colourScheme[1], alpha=0.3)

plot(fig_s2b['residue'].values, fig_s2b['hur332p_betapropensity'].values,
     color=colourScheme[2], label=system_labels[2])
fill_between(fig_s2b['residue'].values,
              fig_s2b['hur332p_betapropensity'].values - fig_s2b['hur332p_betapropensity_sem'].values,
              fig_s2b['hur332p_betapropensity'].values + fig_s2b['hur332p_betapropensity_sem'].values,
              color=colourScheme[2], alpha=0.3)

xlabel('Residue (Rhesus Numbering)', fontsize=25)
ylabel(r'$\beta$ Propensity', fontsize=25)
xticks(fontsize=25)
yticks(fontsize=25)
legend(fontsize=25, loc='right')
title('Figure S2B: Beta Propensity', fontsize=20)

axvspan(326, 349, color='purple', alpha=0.2)
axvspan(381, 398, color='violet', alpha=0.2)
axvspan(419, 428, color='pink', alpha=0.2)
axvspan(483, 489, color='red', alpha=0.1)

tight_layout()
show()

# Figure S2C: Human V1 beta propensity only
fig_s2c = load_fig('fig_s2c.csv')
figure(figsize=(20, 6))

plot(fig_s2c['residue'].values, fig_s2c['hu_betapropensity'].values,
     color=colourScheme[1], label=system_labels[1])
fill_between(fig_s2c['residue'].values,
              fig_s2c['hu_betapropensity'].values - fig_s2c['hu_betapropensity_sem'].values,
              fig_s2c['hu_betapropensity'].values + fig_s2c['hu_betapropensity_sem'].values,
              color=colourScheme[1], alpha=0.3)

plot(fig_s2c['residue'].values, fig_s2c['hur332p_betapropensity'].values,
     color=colourScheme[2], label=system_labels[2])
fill_between(fig_s2c['residue'].values,
              fig_s2c['hur332p_betapropensity'].values - fig_s2c['hur332p_betapropensity_sem'].values,
              fig_s2c['hur332p_betapropensity'].values + fig_s2c['hur332p_betapropensity_sem'].values,
              color=colourScheme[2], alpha=0.3)

xlabel('Residue', fontsize=25)
ylabel(r'$\beta$ Propensity', fontsize=25)
xticks(fontsize=25)
yticks(fontsize=25)
legend(fontsize=25, loc='upper left')
title('Figure S2C: Human V1 Beta Propensity', fontsize=20)
ylim(0, 0.6)
xlim(324, 347)

tight_layout()
show()

## Figures S4A-B: Chemical Shift Validation

In [ ]:
# Figure S4A: CA chemical shifts
fig_s4a = load_fig('fig_s4a.csv')
figure(figsize=(12, 12))
plot(arange(40, 70), arange(40, 70), color='red', linewidth=2)
plot(fig_s4a['expshift'].values, fig_s4a['predhsift'].values, 'o', color='black')
errorbar(fig_s4a['expshift'].values, fig_s4a['predhsift'].values,
         yerr=fig_s4a['yerr'].values, fmt='o', color='black', alpha=0.6, elinewidth=1)
xlabel('Experimental Chemical Shift (ppm)', fontsize=25)
ylabel('Predicted Chemical Shift (ppm)', fontsize=25)
title('Figure S4A: CA Chemical Shift', fontsize=20)
xticks(fontsize=25)
yticks(fontsize=25)
grid(alpha=0.2)
tight_layout()
show()

# Figure S4B: CB chemical shifts
fig_s4b = load_fig('fig_s4b.csv')
figure(figsize=(12, 12))
plot(arange(10, 80), arange(10, 80), color='red', linewidth=2)
plot(fig_s4b['expshift'].values, fig_s4b['predhsift'].values, 'o', color='black')
errorbar(fig_s4b['expshift'].values, fig_s4b['predhsift'].values,
         yerr=fig_s4b['yerr'].values, fmt='o', color='black', alpha=0.6, elinewidth=1)
xlabel('Experimental Chemical Shift (ppm)', fontsize=25)
ylabel('Predicted Chemical Shift (ppm)', fontsize=25)
title('Figure S4B: CB Chemical Shift', fontsize=20)
xticks(fontsize=25)
yticks(fontsize=25)
grid(alpha=0.2)
tight_layout()
show()

## Figures S5: Disorder Prediction

In [ ]:
# Figure S5: Human disorder
fig_s5_human = load_fig('fig_s5_human.csv')
figure(figsize=(14, 6))
plot(fig_s5_human['residues'].values, fig_s5_human['metapredict'].values,
     label='MetaPredict Disorder', linewidth=2.5)
plot(fig_s5_human['residues'].values, fig_s5_human['alphafold'].values,
     label='1 - pLDDT/100', linewidth=2.5)
xlabel('Residue', fontsize=15)
ylabel('Disorder Score', fontsize=15)
title('Figure S5 (Human): Predicted Disorder', fontsize=18)
ylim(0, 1.05)
legend(fontsize=12)
grid(alpha=0.2)
tight_layout()
show()

# Figure S5: Rhesus disorder
fig_s5_rhesus = load_fig('fig_s5_rhesus.csv')
figure(figsize=(14, 6))
plot(fig_s5_rhesus['residues'].values, fig_s5_rhesus['metapredict'].values,
     label='MetaPredict Disorder', linewidth=2.5)
plot(fig_s5_rhesus['residues'].values, fig_s5_rhesus['alphafold'].values,
     label='1 - pLDDT/100', linewidth=2.5)
xlabel('Residue', fontsize=15)
ylabel('Disorder Score', fontsize=15)
title('Figure S5 (Rhesus): Predicted Disorder', fontsize=18)
ylim(0, 1.05)
legend(fontsize=12)
grid(alpha=0.2)
tight_layout()
show()

## Figures S7A-S7D: Radius of Gyration Distributions

In [ ]:
for fig_num, (fname, title_str) in enumerate([
    ('fig_s7a.csv', 'Figure S7A: V1 Loop Rg Distribution'),
    ('fig_s7b.csv', 'Figure S7B: V2 Loop Rg Distribution'),
    ('fig_s7c.csv', 'Figure S7C: V3 Loop Rg Distribution'),
    ('fig_s7d.csv', 'Figure S7D: V1-V2-V3 Rg Distribution'),
]):
    df = load_fig(fname)
    figure(figsize=(16, 6))

    binrange = df['rgs'].values

    plot(binrange, df['human_probability_density'].values, color=colourScheme[1],
         linewidth=4, label='Human WT')
    fill_between(binrange, df['human_probability_density'].values - df['human_probability_density_err'].values,
                  df['human_probability_density'].values + df['human_probability_density_err'].values,
                  color=colourScheme[1], alpha=0.3)

    plot(binrange, df['humanr332p_probability_density'].values, color=colourScheme[2],
         linewidth=4, label='Human R332P')
    fill_between(binrange, df['humanr332p_probability_density'].values - df['humanr332p_probability_density_err'].values,
                  df['humanr332p_probability_density'].values + df['humanr332p_probability_density_err'].values,
                  color=colourScheme[2], alpha=0.3)

    xlabel('Radius of Gyration ($\AA$)', fontsize=25)
    ylabel('Probability Density', fontsize=25)
    xticks(fontsize=25)
    yticks(fontsize=25)
    legend(fontsize=25)
    title(title_str, fontsize=20)

    tight_layout()
    show()

## Figure S8: Contact Probability Difference Map

In [ ]:
fig_s8 = load_matrix('fig_s8.csv')
figure(figsize=(10, 10))
cmap_range = np.max(np.abs(fig_s8[~np.isnan(fig_s8)]))
imshow(fig_s8, cmap=custom_cmap, origin='lower', vmin=-cmap_range, vmax=cmap_range)
colorbar()
title('Figure S8: PRYSPRY Contact Prob Difference (R332P - WT)', fontsize=20)
xlabel('Residue', fontsize=15)
ylabel('Residue', fontsize=15)
tight_layout()
show()


## Figures S9 & S10: Complex & Monomer-specific Contact Maps

In [ ]:
# Figure S9
fig_s9_base = load_fig('fig_s9_base.csv')
fig_s9 = load_fig('fig_s9.csv')

figure(figsize=(12, 12))
if 'pc1' in fig_s9_base.columns and 'pc2' in fig_s9_base.columns:
    scatter(fig_s9_base['pc1'].values, fig_s9_base['pc2'].values, c='gray', alpha=0.15, s=5, label='Base loop ensemble')

pc1_col = 'complex_pc1' if 'complex_pc1' in fig_s9.columns else fig_s9.columns[0]
pc2_col = 'complex_pc2' if 'complex_pc2' in fig_s9.columns else fig_s9.columns[1]
scatter(fig_s9[pc1_col].values, fig_s9[pc2_col].values, c='black', alpha=0.8, s=8, label='Complex trajectories')

xlabel('PC1', fontsize=25)
ylabel('PC2', fontsize=25)
title('Figure S9: Complex in Loop PC Space', fontsize=20)
legend(fontsize=15)
xticks(fontsize=20)
yticks(fontsize=20)
tight_layout()
show()

# Figure S10A-E: Monomer-specific contact maps  
for suffix in ['a', 'b', 'c', 'd', 'e']:
    df = load_matrix(f'fig_s10{suffix}.csv')
    figure(figsize=(10, 10))
    imshow(df, cmap='binary', origin='lower', vmin=0, vmax=1)
    colorbar()
    monomer_ids = {'a': 3, 'b': 4, 'c': 8, 'd': 13, 'e': 18}
    title(f'Figure S10{suffix.upper()}: Monomer {monomer_ids[suffix]} Contacts', fontsize=20)
    xlabel('CA Residue', fontsize=15)
    ylabel('PRYSPRY Residue', fontsize=15)
    tight_layout()
    show()

## Figures S11A-B: 1D Contact Propensity

In [ ]:
# Figure S11A: PRYSPRY contact propensity
fig_s11a = load_fig('fig_s11a.csv')
figure(figsize=(14, 5))
plot(fig_s11a['residues_rh'].values, fig_s11a['contactpropensity'].values,
     linewidth=2.5, color='#332288')
xlabel('PRYSPRY Residue (Rhesus)', fontsize=25)
ylabel('Contact Propensity', fontsize=25)
title('Figure S11A: PRYSPRY-Capsid Propensity', fontsize=20)
xticks(fontsize=20)
yticks(fontsize=20)
grid(alpha=0.2)
tight_layout()
show()

# Figure S11B: Capsid contact propensity
fig_s11b = load_fig('fig_s11b.csv')
figure(figsize=(14, 5))
plot(fig_s11b['residues_capsid'].values, fig_s11b['contactpropensity'].values,
     linewidth=2.5, color='#332288')
xlabel('Capsid Residue', fontsize=25)
ylabel('Contact Propensity', fontsize=25)
title('Figure S11B: Capsid-PRYSPRY Propensity', fontsize=20)
xticks(fontsize=20)
yticks(fontsize=20)
grid(alpha=0.2)
tight_layout()
show()

## Notes

This notebook recreates all main and supplementary figures from `FigureOnly.ipynb` using Zenodo CSV data. Styling, fonts, colors, layouts, and annotations might not exactly match because it was ported by an LLM
